In [2]:
# StyleDNA -- Phase 3: Modeling
# Importing libraries

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

Libraries loaded!


In [4]:
# Load cleaned dataset
df = pd.read_csv('../data/cleaned/styles_cleaned.csv')

print("Dataset loaded!")
print("Rows:", df.shape[0])
df.head()

Dataset loaded!
Rows: 44077


,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt


In [5]:
# Select the columns that define style
features = ['gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'usage']

# Encode text columns into numbers
df_model = df[features].copy()

le = LabelEncoder()
for col in features:
    df_model[col] = le.fit_transform(df_model[col])

print("Encoding complete!")
df_model.head()

Encoding complete!


,gender,masterCategory,subCategory,articleType,baseColour,season,usage
0,2,1,38,103,25,0,0
1,2,1,6,56,2,2,0
2,4,0,42,139,37,3,0
3,2,1,6,127,1,0,0
4,2,1,38,133,13,2,0


In [6]:
# Builds K-Means clustering model
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans.fit(df_model)

# Add cluster labels back to original dataframe
df['style_cluster'] = kmeans.labels_

print("Model trained!")
print("\nItems per style cluster:")
print(df['style_cluster'].value_counts().sort_index())

Model trained!

Items per style cluster:
style_cluster
0    9779
1    8560
2    8550
3    8039
4    9149
Name: count, dtype: int64


In [7]:
# Measure accuracy with silhouette score
score = silhouette_score(df_model, kmeans.labels_, sample_size=5000, random_state=42)
print(f"Silhouette Score: {score:.4f}")
print("\nScore guide:")
print("Close to 1.0 = well separated clusters")
print("Close to 0.0 = overlapping clusters")
print("Negative = items may be in wrong clusters")

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Silhouette Score: 0.3816

Score guide:
Close to 1.0 = well separated clusters
Close to 0.0 = overlapping clusters
Negative = items may be in wrong clusters


In [9]:
# What's in each cluster?
for i in range(5):
    print(f"\n--- Cluster {i} ---")
    cluster_items = df[df['style_cluster'] == i]
    print("Top article types:")
    print(cluster_items['articleType'].value_counts().head(3))
    print("Top colours:")
    print(cluster_items['baseColour'].value_counts().head(3))
    print("Top usage:")
    print(cluster_items['usage'].value_counts().head(3))


--- Cluster 0 ---
Top article types:
articleType
Kurtas      1844
Handbags    1759
Heels       1323
Name: count, dtype: int64
Top colours:
baseColour
Black    2088
Blue     1211
Brown    1157
Name: count, dtype: int64
Top usage:
usage
Casual    6755
Ethnic    2245
Formal     634
Name: count, dtype: int64

--- Cluster 1 ---
Top article types:
articleType
Shirts          2003
Sunglasses       970
Sports Shoes     955
Name: count, dtype: int64
Top colours:
baseColour
Black    2257
Blue     1390
Grey      846
Name: count, dtype: int64
Top usage:
usage
Casual    6148
Sports    1248
Formal     590
Name: count, dtype: int64

--- Cluster 2 ---
Top article types:
articleType
Casual Shoes    2845
Briefs           847
Belts            813
Name: count, dtype: int64
Top colours:
baseColour
Black    2254
Brown     897
White     870
Name: count, dtype: int64
Top usage:
usage
Casual    7695
Formal     279
Ethnic     277
Name: count, dtype: int64

--- Cluster 3 ---
Top article types:
articleType
Tshir

In [10]:
# Give clusters style names
cluster_names = { 
    0: 'Ethnic & Feminine',
    1: 'Street & Active',
    2: 'Classic Minimalist',
    3: 'Everyday Casual',
    4: 'Bold & Bright'
}

df['style_profile'] = df['style_cluster'].map(cluster_names)

print("Style profiles assigned!")
print(df['style_profile'].value_counts())

Style profiles assigned!
style_profile
Ethnic & Feminine     9779
Bold & Bright         9149
Street & Active       8560
Classic Minimalist    8550
Everyday Casual       8039
Name: count, dtype: int64


In [13]:
# Recommendation function
def get_outfit_recommendations(style_profile, num_recomendations=5):
    # Filter items matching the style profile
    profile_items = df[df['style_profile'] == style_profile]

    # Get one item from each key category
    tops = profile_items[profile_items['subCategory'] == 'Topwear'].sample(
        min(2, len(profile_items[profile_items['subCategory'] == 'Topwear'])))
    bottoms = profile_items[profile_items['subCategory'] == 'Bottomwear'].sample(
        min(1, len(profile_items[profile_items['subCategory'] == 'Bottomwear'])))
    shoes = profile_items[profile_items['subCategory'] == 'Shoes'].sample(
        min(1, len(profile_items[profile_items['subCategory'] == 'Shoes'])))
    accessories = profile_items[profile_items['subCategory'] == 'Accessories'].sample(
        min(1, len(profile_items[profile_items['subCategory'] == 'Accessories'])))

    outfit = pd.concat([tops, bottoms, shoes, accessories])

    return outfit[['productDisplayName', 'articleType', 'baseColour', 'usage', 'style_profile']]

# Testing
print("Sample outfit for Bold & Bright: \n")
print(get_outfit_recommendations('Bold & Bright'))

Sample outfit for Bold & Bright: 

                               productDisplayName       articleType  \
40274                 Arrow Woman Striped Red Top              Tops   
29551       Wrangler Women Chervon Yellow T-shirt           Tshirts   
23927     Arrow Sport Men Dark Navy Blue Trousers          Trousers   
11711  ADIDAS Women Vanquish5 Silver Sports Shoes      Sports Shoes   
1368              ADIDAS Unisex Red Organizer Bag  Travel Accessory   

      baseColour   usage  style_profile  
40274        Red  Casual  Bold & Bright  
29551     Yellow  Casual  Bold & Bright  
23927  Navy Blue  Casual  Bold & Bright  
11711     Silver  Sports  Bold & Bright  
1368         Red  Casual  Bold & Bright  
